# 10 Partitioning, Z-Ordering and Liquid Clustering

## Goal

<div style="background:#E8F1FF;border-left:5px solid #2F6FED;padding:10px 14px;border-radius:6px;color:#1b2a41;">
<b>💡 Goal</b><br>
Understand <b>data skipping</b> (how Delta uses per-file min/max statistics to avoid reading files), and the three ways to lay data out so skipping works: <b>partitioning</b>, <b>Z-ordering</b> and <b>liquid clustering</b>. Know when to use which, and why Databricks now recommends liquid clustering for most tables.
</div>

## Why it matters

<div style="background:#E7F7EE;border-left:5px solid #1E9E5A;padding:10px 14px;border-radius:6px;color:#14391f;">
<b>✅ Why it matters</b><br>
On large tables, the fastest query is the one that reads the fewest files. Layout decides that: the same filter can read 100% or 1% of a table depending on how rows are grouped into files. "Partitioning vs Z-ordering vs liquid clustering" is a frequent interview and exam topic.
</div>

## Concept

<div style="background:#E8F1FF;border-left:5px solid #2F6FED;padding:10px 14px;border-radius:6px;color:#1b2a41;">
<b>💡 Data skipping</b><br>
Delta stores, for every data file, the <b>min and max</b> of the first 32 columns (configurable) in the transaction log. For a filter such as <code>customer_id = 1234</code>, Delta reads only files whose <code>[min, max]</code> range can contain 1234.<br><br>
Skipping only helps if the values are <b>clustered</b>: if every file contains customers from 1 to 100,000, every range covers 1234 and nothing is skipped.
</div>

| | Partitioning | Z-ordering | Liquid clustering |
|---|---|---|---|
| How | One folder per value (`PARTITIONED BY`) | `OPTIMIZE t ZORDER BY (cols)` rewrites files so nearby values share files | `CLUSTER BY (cols)`, maintained incrementally by `OPTIMIZE` |
| Good for | Low-cardinality columns with large partitions (≥ 1 GB), such as date on very big tables | High-cardinality filter columns on older runtimes | Most new tables, including high-cardinality and multi-column filters |
| Change the columns later | Rewrite the whole table | Run a new `ZORDER` (full rewrite) | `ALTER TABLE ... CLUSTER BY (...)`, no immediate rewrite |
| Incremental | n/a | No: each `ZORDER` rewrites the data | Yes: only new or unclustered data is reorganized |
| Combine with partitioning | n/a | Yes (Z-order within partitions) | No (liquid replaces partitioning) |
| Databricks recommendation | Only for very large tables with an obvious partition column | Legacy | ✅ Default choice (or `CLUSTER BY AUTO`) |

## Real-world example

<div style="background:#FFF4DB;border-left:5px solid #E5A100;padding:10px 14px;border-radius:6px;color:#4a3500;">
<b>🏦 Real-world example</b><br>
A 3 TB <code>transactions</code> table was partitioned by <code>transaction_date</code> and Z-ordered by <code>account_id</code>. Analysts filtering by <code>merchant_id</code> got no skipping at all, and changing the Z-order meant rewriting 3 TB. After moving to liquid clustering on <code>(transaction_date, account_id)</code> and later adding <code>merchant_id</code> with <code>ALTER TABLE ... CLUSTER BY</code>, new data was clustered incrementally and queries on all three columns skipped most files, with no full rewrite.
</div>

## Setup

<div style="background:#E8F1FF;border-left:5px solid #2F6FED;padding:10px 14px;border-radius:6px;color:#1b2a41;">
<b>💡 What it does</b><br>
Creates a 2-million-row table of payments in random order, spread over 16 files, and a helper that computes each file's min and max <code>customer_id</code>, the same check Delta does with its statistics.
</div>
<br>
<div style="background:#E7F7EE;border-left:5px solid #1E9E5A;padding:10px 14px;border-radius:6px;color:#14391f;">
<b>✅ Expected output</b><br>
16 files, and every file's range covers the whole customer domain.
</div>

In [0]:
DB = "workspace.delta_lake"
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {DB}")

In [0]:
from pyspark.sql import functions as F

payments = spark.range(2_000_000).select(
    (F.rand(seed=1) * 100_000).cast("int").alias("customer_id"),
    F.expr("date_add(date'2024-01-01', cast(rand(2) * 365 AS INT))").alias("pay_date"),
    F.round(F.rand(seed=3) * 500, 2).alias("amount"),
)

def file_ranges(table, col="customer_id"):
    """min/max of a column per data file, read through the _metadata column."""
    return (spark.table(table)
            .groupBy(F.col("_metadata.file_path").alias("file"))
            .agg(F.min(col).alias("lo"), F.max(col).alias("hi")))

def skipping_report(table, value, col="customer_id"):
    r = file_ranges(table, col)
    total = r.count()
    needed = r.filter((F.col("lo") <= value) & (F.col("hi") >= value)).count()
    print(f"{table.split('.')[-1]:<22} files: {total:>3} | files that may contain {col}={value}: {needed:>3} | skippable: {1 - needed / total:.0%}")

random_t = f"{DB}.payments_random"
payments.repartition(16).write.mode("overwrite").saveAsTable(random_t)
skipping_report(random_t, 1234)

## 1. No layout, no skipping

<div style="background:#E8F1FF;border-left:5px solid #2F6FED;padding:10px 14px;border-radius:6px;color:#1b2a41;">
<b>💡 What it does</b><br>
Shows the per-file ranges of the randomly ordered table.<br><br>
<b>Why it matters</b><br>This is the default after appends from many sources: every file holds every kind of customer. Statistics exist, but they can't rule out any file.
</div>
<br>
<div style="background:#E7F7EE;border-left:5px solid #1E9E5A;padding:10px 14px;border-radius:6px;color:#14391f;">
<b>✅ Expected output</b><br>
Each file's <code>lo</code> is near 0 and <code>hi</code> near 99,999.
</div>

In [0]:
file_ranges(random_t).orderBy("lo").show(5, truncate=40)

## 2. Z-ordering

<div style="background:#E8F1FF;border-left:5px solid #2F6FED;padding:10px 14px;border-radius:6px;color:#1b2a41;">
<b>💡 What it does</b><br>
Runs <code>OPTIMIZE ... ZORDER BY (customer_id)</code> on a copy of the table, with a small target file size so the demo produces several files, and measures skipping again.<br><br>
<b>Why it matters</b><br>Z-ordering rewrites the files so that rows with nearby values end up in the same files. Each file then covers a narrow range, and a filter on <code>customer_id</code> can skip almost every file. Z-ordering on several columns uses a space-filling curve, so it works for filters on any of them, though less sharply than on one.
</div>
<br>
<div style="background:#FFF4DB;border-left:5px solid #E5A100;padding:10px 14px;border-radius:6px;color:#4a3500;">
<b>🆓 Free Edition note</b><br>
The target file size is set with the Databricks table property <code>delta.targetFileSize</code>, and with a session setting for open-source Delta. Whichever applies is used. With real data sizes you don't need to set it: <code>OPTIMIZE</code> aims for files of about 1 GB.
</div>
<br>
<div style="background:#E7F7EE;border-left:5px solid #1E9E5A;padding:10px 14px;border-radius:6px;color:#14391f;">
<b>✅ Expected output</b><br>
The <code>OPTIMIZE</code> metrics show the files removed and added. Afterwards, only 1 (or very few) of the files can contain <code>customer_id = 1234</code>.
</div>

In [0]:
zorder_t = f"{DB}.payments_zorder"
spark.table(random_t).write.mode("overwrite").saveAsTable(zorder_t)

def small_files_for_demo(table):
    for apply in [lambda: spark.sql(f"ALTER TABLE {table} SET TBLPROPERTIES ('delta.targetFileSize' = '2mb')"),
                  lambda: spark.conf.set("spark.databricks.delta.optimize.maxFileSize", str(2 * 1024 * 1024))]:
        try:
            apply()
        except Exception:
            pass

small_files_for_demo(zorder_t)
spark.sql(f"OPTIMIZE {zorder_t} ZORDER BY (customer_id)").select("metrics.numFilesRemoved", "metrics.numFilesAdded").show()
skipping_report(zorder_t, 1234)
file_ranges(zorder_t).orderBy("lo").show(5)

## 3. Liquid clustering

<div style="background:#E8F1FF;border-left:5px solid #2F6FED;padding:10px 14px;border-radius:6px;color:#1b2a41;">
<b>💡 What it does</b><br>
Creates a table with <code>CLUSTER BY (customer_id)</code>, loads the same data, runs <code>OPTIMIZE</code>, and measures skipping. Then it changes the clustering keys with <code>ALTER TABLE</code>.<br><br>
<b>Why it matters</b><br>Liquid clustering gives the same skipping benefits as Z-ordering, but:
<ul><li>You declare it once on the table, and <code>OPTIMIZE</code> (or predictive optimization) keeps it clustered <b>incrementally</b></li><li>You can change the keys at any time without rewriting existing data</li><li>It replaces partitioning and Z-ordering, so it can't be combined with them</li></ul>
</div>
<br>
<div style="background:#E7F7EE;border-left:5px solid #1E9E5A;padding:10px 14px;border-radius:6px;color:#14391f;">
<b>✅ Expected output</b><br>
<code>clusteringColumns = [customer_id]</code>, then only 1 (or very few) files may contain <code>customer_id = 1234</code>. After the <code>ALTER</code>, the clustering columns become <code>[customer_id, pay_date]</code>.
</div>

In [0]:
liquid_t = f"{DB}.payments_liquid"
spark.sql(f"CREATE OR REPLACE TABLE {liquid_t} (customer_id INT, pay_date DATE, amount DOUBLE) CLUSTER BY (customer_id)")
small_files_for_demo(liquid_t)
spark.table(random_t).write.mode("append").saveAsTable(liquid_t)
spark.sql(f"OPTIMIZE {liquid_t}").select("metrics.numFilesAdded").show()

spark.sql(f"DESCRIBE DETAIL {liquid_t}").select("clusteringColumns", "numFiles").show()
skipping_report(liquid_t, 1234)

spark.sql(f"ALTER TABLE {liquid_t} CLUSTER BY (customer_id, pay_date)")
spark.sql(f"DESCRIBE DETAIL {liquid_t}").select("clusteringColumns").show()

## 4. Automatic liquid clustering (Databricks)

<div style="background:#E8F1FF;border-left:5px solid #2F6FED;padding:10px 14px;border-radius:6px;color:#1b2a41;">
<b>💡 What it does</b><br>
Shows <code>CLUSTER BY AUTO</code>, which lets Databricks choose and evolve the clustering keys from the query workload.<br><br>
<b>Why it matters</b><br>With predictive optimization, Databricks watches which columns your queries filter on, chooses clustering keys, and maintains them. It's the lowest-effort layout for Unity Catalog managed tables.
</div>
<br>
<div style="background:#FFF4DB;border-left:5px solid #E5A100;padding:10px 14px;border-radius:6px;color:#4a3500;">
<b>🆓 Free Edition note</b><br>
<code>CLUSTER BY AUTO</code> needs Unity Catalog managed tables and predictive optimization. If it isn't available in your workspace, the cell prints a note.
</div>
<br>
<div style="background:#E7F7EE;border-left:5px solid #1E9E5A;padding:10px 14px;border-radius:6px;color:#14391f;">
<b>✅ Expected output</b><br>
The table is created with automatic clustering, or a note explains it isn't available.
</div>

In [0]:
auto_t = f"{DB}.payments_auto"
try:
    spark.sql(f"CREATE OR REPLACE TABLE {auto_t} (customer_id INT, pay_date DATE, amount DOUBLE) CLUSTER BY AUTO")
    display(spark.sql(f"DESCRIBE DETAIL {auto_t}").select("clusteringColumns", "properties"))
    spark.sql(f"DROP TABLE {auto_t}")
except Exception as e:
    print("CLUSTER BY AUTO not available here:", type(e).__name__)

## 5. Partitioning, revisited for Delta

<div style="background:#E8F1FF;border-left:5px solid #2F6FED;padding:10px 14px;border-radius:6px;color:#1b2a41;">
<b>💡 What it does</b><br>
Creates a table partitioned by month and compares partition pruning with data skipping inside partitions.<br><br>
<b>Why it matters</b><br>Partitioning still prunes whole folders, and it works well when the partition column is low-cardinality and always filtered (for example a date on a multi-terabyte table). But it can't help filters on other columns, and over-partitioning creates small files (lesson 01_25). Inside each partition, data skipping on other columns still needs clustered data.
</div>
<br>
<div style="background:#E7F7EE;border-left:5px solid #1E9E5A;padding:10px 14px;border-radius:6px;color:#14391f;">
<b>✅ Expected output</b><br>
12 partitions. Filtering one month reads only that partition's files, but the files inside it still cover all customers.
</div>

In [0]:
part_t = f"{DB}.payments_by_month"
(payments.withColumn("pay_month", F.date_format("pay_date", "yyyy-MM"))
         .write.mode("overwrite").partitionBy("pay_month").saveAsTable(part_t))

detail = spark.sql(f"DESCRIBE DETAIL {part_t}").first()
print("partition columns:", detail["partitionColumns"], "| files:", detail["numFiles"])
march = spark.table(part_t).filter("pay_month = '2024-03'")
print("files read for March:", march.select("_metadata.file_path").distinct().count())
march.explain()

## Under the hood

```
query: WHERE customer_id = 1234
  1. partition pruning   (partitioned tables)  -> drop folders that can't match
  2. data skipping       (stats in the log)    -> drop files whose [min, max] can't contain 1234
  3. Parquet row groups  (footer stats)        -> drop row groups inside the remaining files
  4. read what's left

Z-ORDER / liquid clustering make step 2 effective by giving each file a narrow range.
```

<span style="background:#2F6FED;color:white;padding:2px 10px;border-radius:12px;font-size:12px;">Transformation</span> skipping happens during planning, from the log. Skipped files are never opened.

Statistics are collected for the first 32 columns by default (`delta.dataSkippingNumIndexedCols`, or `delta.dataSkippingStatsColumns` to choose them). Put the columns you filter on among the indexed ones.

In [0]:
spark.table(liquid_t).filter("customer_id = 1234").explain()

In [0]:
for t in [random_t, zorder_t, liquid_t, part_t]:
    spark.sql(f"DROP TABLE IF EXISTS {t}")
try:
    spark.conf.unset("spark.databricks.delta.optimize.maxFileSize")
except Exception:
    pass

## Common mistakes and troubleshooting log

<div style="background:#FDEAEA;border-left:5px solid #D64545;padding:10px 14px;border-radius:6px;color:#4a1414;">
<b>⛔ Problem: filters on a column are slow even though statistics exist</b><br>
The data isn't clustered by that column, so every file's range covers the value. Cluster (liquid or Z-order) by it.<br><br>
<b>⛔ Problem: Z-ordering by a partition column</b><br>Not allowed and pointless: each partition already holds one value. Z-order by other columns.<br><br>
<b>⛔ Problem: Z-ordering by many columns</b><br>Effectiveness drops with each column. Pick the 1–4 most filtered columns.<br><br>
**⛔ Problem: <code>CLUSTER BY</code> fails on a partitioned table**<br>Liquid clustering replaces partitioning. Create a new table without <code>PARTITIONED BY</code>.<br><br>
<b>⛔ Problem: the filter column has no statistics</b><br>It's beyond the first 32 columns. Reorder the columns, or set <code>delta.dataSkippingStatsColumns</code>.<br><br>
<b>✍️ My own problems</b><br>(add yours here)
</div>

## Interview Q&A

<div style="background:#F1E9FF;border-left:5px solid #7A4DD8;padding:10px 14px;border-radius:6px;color:#2d1a57;">
<b>🎤 1. What is data skipping in Delta Lake?</b><br>
Delta stores min and max values per column for each data file in the transaction log. At query time it compares filters with those ranges and skips files that can't contain matching rows, without opening them.<br><br>

<b>🎤 2. What does Z-ordering do?</b><br>
<code>OPTIMIZE ... ZORDER BY (cols)</code> rewrites data so that rows with similar values of the chosen columns are stored in the same files, using a space-filling curve for several columns. Each file then covers narrow ranges, which makes data skipping effective.<br><br>

<b>🎤 3. What is liquid clustering and why is it recommended?</b><br>
A Delta layout feature declared with <code>CLUSTER BY</code>. Data is clustered by the chosen keys and maintained incrementally by <code>OPTIMIZE</code> or predictive optimization. Keys can be changed without rewriting the table. It replaces both partitioning and Z-ordering, handles high-cardinality columns well, and with <code>CLUSTER BY AUTO</code> Databricks can even choose the keys.<br><br>

<b>🎤 4. When would you still partition a Delta table?</b><br>
For very large tables (terabytes) with a low-cardinality column that almost every query filters on, typically a date, so that each partition is at least around 1 GB. Otherwise liquid clustering is the better default.<br><br>

<b>🎤 5. Partitioning vs Z-ordering?</b><br>
Partitioning splits data into folders by column value and prunes whole folders, which suits low-cardinality columns. Z-ordering co-locates values inside files to improve file-level skipping, which suits high-cardinality columns. They can be combined, with Z-ordering within partitions, but liquid clustering now replaces both.
</div>

## Certification corner

<div style="background:#E6F6F7;border-left:5px solid #0E8C8C;padding:10px 14px;border-radius:6px;color:#0b3536;">
<b>🎓 Exam-style questions (Data Engineer Associate)</b><br><br>
<b>Q1. A Delta table is frequently filtered by <code>customer_id</code>, which has millions of distinct values. Which approach improves data skipping and lets the team change the clustering columns later without rewriting the table?</b><br>
A. Partition by <code>customer_id</code><br>
B. <code>OPTIMIZE ... ZORDER BY (customer_id)</code><br>
C. Liquid clustering with <code>CLUSTER BY (customer_id)</code><br>
D. Increase the number of shuffle partitions<br>
<details><summary><b>Show answer</b></summary><b>C.</b> A creates millions of tiny partitions, and B works but needs a full rewrite whenever the columns change.</details><br><br>

<b>Q2. Where does Delta Lake store the statistics used for data skipping?</b><br>
A. In a separate Hive metastore table<br>
B. In the transaction log, per data file<br>
C. In the driver's memory only<br>
D. In each partition folder name<br>
<details><summary><b>Show answer</b></summary><b>B.</b></details><br><br>

<b>Q3. Which command changes the clustering columns of a liquid clustered table?</b><br>
A. <code>OPTIMIZE t ZORDER BY (c)</code><br>
B. <code>ALTER TABLE t CLUSTER BY (c)</code><br>
C. <code>ALTER TABLE t PARTITIONED BY (c)</code><br>
D. <code>REORG TABLE t</code><br>
<details><summary><b>Show answer</b></summary><b>B.</b></details>
</div>

## Practice problem

<div style="background:#FFEFE5;border-left:5px solid #F07D2E;padding:10px 14px;border-radius:6px;color:#4a2300;">
<b>🧪 Without copying from above:</b>
<ol><li>Create a liquid clustered table of <code>payments</code> clustered by <code>pay_date</code>, and measure how many files may contain <code>pay_date = '2024-06-15'</code></li><li>Compare it with the random layout for the same filter</li><li>Change the clustering keys to <code>(pay_date, customer_id)</code> and run <code>OPTIMIZE</code> again</li><li>In one sentence: why does data skipping not help on the randomly ordered table even though statistics exist?</li></ol>
</div>

In [0]:
# Your solution for the practice problem goes here.
# Add a comment above each step explaining what it does.

## Solution

<div style="background:#FFF4DB;border-left:5px solid #E5A100;padding:10px 14px;border-radius:6px;color:#4a3500;">
<b>⚠️ Try it yourself first</b><br>
Only look at this after you have a working answer of your own.
</div>

In [0]:
from datetime import date

# 1. Liquid clustering by date
t_date = f"{DB}.payments_by_date_lc"
spark.sql(f"CREATE OR REPLACE TABLE {t_date} (customer_id INT, pay_date DATE, amount DOUBLE) CLUSTER BY (pay_date)")
small_files_for_demo(t_date)
payments.write.mode("append").saveAsTable(t_date)
spark.sql(f"OPTIMIZE {t_date}")
skipping_report(t_date, date(2024, 6, 15), col="pay_date")

# 2. Random layout for the same filter
rand_t = f"{DB}.payments_random2"
payments.repartition(16).write.mode("overwrite").saveAsTable(rand_t)
skipping_report(rand_t, date(2024, 6, 15), col="pay_date")

# 3. New keys, incremental re-clustering
spark.sql(f"ALTER TABLE {t_date} CLUSTER BY (pay_date, customer_id)")
spark.sql(f"OPTIMIZE {t_date}")
spark.sql(f"DESCRIBE DETAIL {t_date}").select("clusteringColumns", "numFiles").show()

# 4. Every file contains the full range of values, so no file's [min, max] can rule out the filter value.
for t in [t_date, rand_t]:
    spark.sql(f"DROP TABLE {t}")

## Summary and cheat sheet

<div style="background:#E7F7EE;border-left:5px solid #1E9E5A;padding:10px 14px;border-radius:6px;color:#14391f;">
<b>✅ Key takeaways</b>
<ul><li>Data skipping uses per-file min/max statistics from the log. It only works when data is clustered</li><li>Partitioning: folders per value, for low-cardinality, always-filtered columns on very large tables</li><li>Z-ordering: <code>OPTIMIZE ... ZORDER BY</code>, co-locates values, full rewrite each time (legacy)</li><li>Liquid clustering: <code>CLUSTER BY</code>, incremental, keys can change. The recommended default (<code>CLUSTER BY AUTO</code> on Databricks)</li><li>Liquid clustering can't be combined with partitioning or Z-order</li><li>Statistics cover the first 32 columns by default</li></ul>
</div>

| Task | SQL |
|---|---|
| Liquid clustering | `CREATE TABLE t (...) CLUSTER BY (a, b)` |
| Change keys | `ALTER TABLE t CLUSTER BY (c)` |
| Automatic keys | `CREATE TABLE t (...) CLUSTER BY AUTO` |
| Cluster now | `OPTIMIZE t` |
| Z-order | `OPTIMIZE t ZORDER BY (a)` |
| Partition | `CREATE TABLE t (...) PARTITIONED BY (d)` |
| Check | `DESCRIBE DETAIL t` → `clusteringColumns`, `partitionColumns` |

## Git commit

```
git add 03_delta_lake/10_partitioning_zorder_clustering.ipynb
git commit -m "add 03_10 partitioning zorder clustering notebook"
```